# 📈 American Option Demo (QoX)

This notebook demonstrates pricing an **American Put Option** using QoX.

### 🚀 How to use:
1. Run the setup cell below
2. Edit the code in the next cell ✏️
3. Click ▶ Run to see results


In [ ]:
# 🔧 Setup
!pip install qox -q

print("✅ QoX installed. You're ready to go!")

## ✏️ Editable Example

**👉 Modify anything below and re-run the cell**

Try changing:
- spot price
- volatility
- interest rate
- number of FDM steps


In [ ]:
# 👇 EDIT THIS CELL

from datetime import datetime, timedelta
import time

# QoX Python API
from qox import *

# -----------------------------
# Configuration
# -----------------------------

config = Config().add_policy(
    InstrumentPolicy()
    .american()
    .put()
    .layer(PricingMethod.FDM(
        FdmConfig(time_steps=20, nodes=500)
    ))
)

# -----------------------------
# Instrument
# -----------------------------

expiry = datetime.utcnow() + timedelta(days=365)

stock_option = StockOption(
    strike=100.0,
    expiry=expiry,
    option_type=OptionType.PUT,
    exercise_style=ExerciseStyle.AMERICAN
)

# -----------------------------
# Market Data
# -----------------------------

spot = 95.0
rate = 0.05
vol = 0.2

market = OptionMarketFrame(
    spot=spot,
    rate_curve=FlatRateCurve(rate),
    vol_surface=FlatVolSurface(vol)
)

# -----------------------------
# Valuation
# -----------------------------

result = stock_option.valuation() \
    .market(market) \
    .config(config) \
    .compute()

# Benchmark
start = time.time()
n = 100

for _ in range(n):
    result = stock_option.valuation() \
        .market(market) \
        .config(config) \
        .compute()

duration = (time.time() - start) / n

# -----------------------------
# Output
# -----------------------------

print(f"Price: {result.price:.8f}")
print(f"Delta: {result.greeks.delta:.8f}")
print(f"Gamma: {result.greeks.gamma:.8f}")
print(f"Theta: {result.greeks.theta:.8f}")

if result.greeks.vega is not None:
    print(f"Vega: {result.greeks.vega:.8f}")

if result.greeks.rho is not None:
    print(f"Rho: {result.greeks.rho:.8f}")

print(f"Time per run: {duration:.6f} sec")